# Zero-Shot & CoT Prompting — Gemini 2.5 Flash

- Uses `google-genai` SDK
- Zero-shot: thinking disabled (`thinking_budget=0`), direct answer
- CoT: thinking enabled, reasoning extracted from `thought` blocks
- Incremental CSV saving every 50 rows
- Resume logic via `(Puzzle_Number, Question_Number)` composite key

## 1. Install & Imports

In [ ]:
!pip install -q google-genai pandas openpyxl tqdm

In [ ]:
import re
import os
import time
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types

## 2. Config — edit here only

In [ ]:
# ── API ───────────────────────────────────────────────────────────────────────
GEMINI_API_KEY = 'REDACTED_USE_ENVIRONMENT_VARIABLE'   # replace or load from env
MODEL_ID       = 'gemini-2.5-flash'

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = '/content/benchmark_originals_for_rerun.csv'
WORKDIR   = './outputs'
os.makedirs(WORKDIR, exist_ok=True)

ZS_OUT  = os.path.join(WORKDIR, 'puzzle_zeroshot_gemini25flash.csv')
COT_OUT = os.path.join(WORKDIR, 'puzzle_cot_gemini25flash.csv')

# ── Generation ────────────────────────────────────────────────────────────────
SAVE_EVERY     = 50
MAX_OUTPUT_ZS  = 256    # zero-shot: just a letter
MAX_OUTPUT_COT = 5000   # CoT: thinking trace can be long

# ── Thinking budget ───────────────────────────────────────────────────────────
# 0     = thinking disabled (zero-shot)
# -1    = dynamic budget, model decides (CoT)
# 1-N   = fixed token budget for thinking
THINKING_BUDGET_ZS  = 0
THINKING_BUDGET_COT = -1

## 3. Init Client

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)
print(f'Client ready. Model: {MODEL_ID}')

## 4. Load Dataset

In [ ]:
sample = pd.read_csv(DATA_PATH)

In [ ]:
# Trim to exactly 100 by dropping 2 from the largest tier
sample = sample.sort_values('difficulty')
largest_tier = sample['difficulty'].value_counts().index[0]
drop_idx = sample[sample['difficulty'] == largest_tier].index[-2:]
sample = sample.drop(drop_idx)

print(f'Final sample: {len(sample)}')
print(sample['difficulty'].value_counts().to_string())

In [ ]:
sample.to_excel("benchmark_100_formatted2.xlsx",index=False)

## 5. Helpers

In [ ]:
# ── Parsing ───────────────────────────────────────────────────────────────────

def extract_answer(text: str) -> str | None:
    """
    Extracts A-E from model output.
    Tries specific patterns first, bare letter last.
    """
    if not isinstance(text, str) or not text.strip():
        return None
    patterns = [
        r'Answer:\s*([A-E])\b',
        r'\*\*([A-E])\*\*',        # bold letter
        r'\b([A-E])\b',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    return None


# ── Incremental saving & resume ───────────────────────────────────────────────

def save_results(records: list, path: str):
    pd.DataFrame(records).to_csv(path, index=False)


def load_completed(path: str, key_cols: list):
    if not os.path.exists(path):
        return [], set()
    done_df   = pd.read_csv(path)
    done_keys = set(zip(*[done_df[c] for c in key_cols]))
    records   = done_df.to_dict('records')
    print(f'  Resuming: {len(records)} rows already in {os.path.basename(path)}')
    return records, done_keys


import time

def generate(system_prompt, user_prompt, max_output_tokens, thinking_budget):
    max_retries = 5
    wait = 30  # start with 30s, doubles each retry

    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=max_output_tokens,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=thinking_budget,
                        include_thoughts=True,
                    ),
                ),
            )
            elapsed = (time.time() - t0) / 60.0

            raw, reasoning = '', None
            for part in response.candidates[0].content.parts:
                if not part.text:
                    continue
                if part.thought:
                    reasoning = part.text.strip()
                else:
                    raw += part.text
            return raw.strip(), reasoning, elapsed

        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise

## 6. System Prompts

In [ ]:
ZERO_SHOT_SYSTEM_PROMPT = """\
You are an expert puzzle solver. Your task is to answer logical multiple-choice puzzles.
Each puzzle consists of three parts: <context>, <clues>, and <question>.

Instructions:
- Read the puzzle carefully.
- Select the single correct answer from the given options (A, B, C, D, or E).
- Output only the correct option letter. Nothing else.
"""

# Thinking is enabled via thinking_budget — prompt stays minimal.
# Gemini will reason internally and we extract the thought block.
COT_SYSTEM_PROMPT = """\
You are an expert puzzle solver. Your task is to answer logical multiple-choice puzzles.
Each puzzle consists of three parts: <context>, <clues>, and <question>.

Instructions:
- Read the puzzle carefully.
- Think through the problem carefully before answering.
- Output only the correct option letter (A, B, C, D, or E). Nothing else.
"""

## 7. Zero-Shot Evaluation
`thinking_budget=0` — no thinking, direct answer only.

In [ ]:
KEY_COLS = ['Puzzle_Number', 'Question_Number']
zs_records, zs_done = load_completed(ZS_OUT, KEY_COLS)

for _, row in tqdm(df.iterrows(), total=len(df), desc='Zero-Shot'):
    key = (row['Puzzle_Number'], row['Question_Number'])
    if key in zs_done:
        continue

    raw, reasoning, elapsed = '', None, 0.0
    answer = None
    try:
        raw, reasoning, elapsed = generate(
            system_prompt      = ZERO_SHOT_SYSTEM_PROMPT,
            user_prompt        = row['Sample'],
            max_output_tokens  = MAX_OUTPUT_ZS,
            thinking_budget    = THINKING_BUDGET_ZS,
        )
        answer = extract_answer(raw)
    except Exception as e:
        print(f'  [Row {_}] Error: {e}')
        raw = f'ERROR: {e}'

    zs_records.append({
        'Puzzle_Number':   row['Puzzle_Number'],
        'Question_Number': row['Question_Number'],
        'Sample':          row['Sample'],
        'Raw_Output':      raw,
        'Reasoning':       reasoning,
        'Model_Answer':    answer,
        'Inference_Min':   round(elapsed, 3),
    })
    zs_done.add(key)

    if len(zs_records) % SAVE_EVERY == 0:
        save_results(zs_records, ZS_OUT)
        print(f'  [Checkpoint] {len(zs_records)} rows saved.')

save_results(zs_records, ZS_OUT)
zs_df = pd.DataFrame(zs_records)
print(f'\nZero-shot complete. {zs_df["Model_Answer"].notna().sum()}/{len(zs_df)} parsed.')
zs_df.head()

## 8. CoT Evaluation
`thinking_budget=-1` — dynamic thinking, reasoning extracted from thought block.

In [ ]:
df = sample

In [ ]:
KEY_COLS = ['Puzzle_Number', 'Question_Number','Sample']
cot_records, cot_done = load_completed(COT_OUT, KEY_COLS)

for _, row in tqdm(df.iterrows(), total=len(df), desc='CoT'):
    key = (row['Puzzle_Number'], row['Question_Number'],row['Sample'])
    if key in cot_done:
        continue

    raw, reasoning, elapsed = '', None, 0.0
    answer = None
    try:
        raw, reasoning, elapsed = generate(
            system_prompt      = COT_SYSTEM_PROMPT,
            user_prompt        = row['Sample'],
            max_output_tokens  = MAX_OUTPUT_COT,
            thinking_budget    = THINKING_BUDGET_COT,
        )
        answer = extract_answer(raw)
    except Exception as e:
        print(f'  [Row {_}] Error: {e}')
        raw = f'ERROR: {e}'

    cot_records.append({
        'Puzzle_Number':   row['Puzzle_Number'],
        'Question_Number': row['Question_Number'],
        'Sample':          row['Sample'],
        'Raw_Output':      raw,
        'Reasoning':       reasoning,
        'Model_Answer':    answer,
        'Inference_Min':   round(elapsed, 3),
    })
    cot_done.add(key)

    if len(cot_records) % SAVE_EVERY == 0:
        save_results(cot_records, COT_OUT)
        print(f'  [Checkpoint] {len(cot_records)} rows saved.')

save_results(cot_records, COT_OUT)
cot_df = pd.DataFrame(cot_records)
print(f'\nCoT complete. {cot_df["Model_Answer"].notna().sum()}/{len(cot_df)} parsed.')
cot_df.head()

## 9. Quick Accuracy Check (if ground truth available)

In [ ]:
# Uncomment if your df has a 'Ground_Truth' column
# for label, result_df in [('Zero-Shot', zs_df), ('CoT', cot_df)]:
#     merged = result_df.merge(
#         df[['Puzzle_Number', 'Question_Number', 'Ground_Truth']],
#         on=['Puzzle_Number', 'Question_Number']
#     )
#     acc = (merged['Model_Answer'] == merged['Ground_Truth']).mean()
#     print(f'{label} accuracy: {acc:.2%}')

In [ ]:
for label, result_df in [('CoT', cot_df)]:
     merged = result_df.merge(
         df[['Puzzle_Number', 'Question_Number', 'Answer']],
         on=['Puzzle_Number', 'Question_Number']
     )
     acc = (merged['Model_Answer'] == merged['Answer']).mean()
     print(f'{label} accuracy: {acc:.2%}')

In [ ]:
def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

sample['difficulty'] = sample['Solved_Percentile'].apply(map_difficulty)

print(f'Total puzzles: {len(sample)}')
print(f'\nDifficulty distribution:')
print(sample['difficulty'].value_counts().to_string())
print(f'\nSolved_Percentile stats:')
print(sample['Solved_Percentile'].describe().round(2).to_string())
print(f'\nMean percentile: {sample["Solved_Percentile"].mean():.1f}%')

# Debug

In [ ]:
from google import genai
from google.genai import types

prompt = "What is the sum of the first 50 prime numbers?"
response = client.models.generate_content(
  model="gemini-2.5-flash",
  contents=prompt,
  config=types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
      include_thoughts=True
    )
  )
)

for part in response.candidates[0].content.parts:
  if not part.text:
    continue
  if part.thought:
    print("Thought summary:")
    print(part.text)
    print()
  else:
    print("Answer:")
    print(part.text)
    print()